# 01. Base de conhecimento e busca
Base com trechos de orientações oficiais (gov.br) sobre alagamentos e enchentes. Aqui conferimos a base e medimos a busca (recall@1 e recall@3).

In [1]:
import sys
sys.path.insert(0, '../src')
import json
import pandas as pd
from avaliar import carregar_casos
from busca import carregar_trechos
t = pd.DataFrame(carregar_trechos())
t.groupby(['fonte', 'url']).size().rename('trechos').reset_index()

,fonte,url,trechos
0,"Como agir: Alagamentos (MDR, 2021)",https://www.gov.br/mdr/pt-br/assuntos/protecao...,37
1,Como agir: Inundações (MDR),https://www.gov.br/mdr/pt-br/assuntos/protecao...,6
2,Como me proteger da leptospirose (Min. da Saúd...,https://www.gov.br/saude/pt-br/acesso-a-inform...,4
3,Medidas de autoproteção em alagamentos (gov.br...,https://www.gov.br/pt-br/noticias/transito-e-t...,14
4,Prevenção da leptospirose em enchentes (Min. d...,https://www.gov.br/pt-br/noticias/saude-e-vigi...,18


In [2]:
t.sample(6, random_state=1)[['id', 'texto']]

,id,texto
63,lepto_medidas-08,"Se precisar ficar na água, utilize botas e luv..."
27,alagamentos_mdr-28,Não use equipamentos elétricos que tenham sido...
31,alagamentos_mdr-32,Mantenha a calma e siga rigorosamente as orien...
69,lepto_medidas-14,"Nas formas mais graves, geralmente aparece ict..."
46,autoprotecao-05,Retire o lixo e coloque-o em áreas de descarte...
47,autoprotecao-06,Procure informações sobre o histórico de alaga...


## Recall da busca
A pergunta de teste está certa se o trecho esperado aparece na posição 1 (recall@1) ou entre os 3 primeiros (recall@3). Comparo a busca por palavras (TF-IDF) com a busca semântica (embeddings bge-m3).

In [3]:
from busca import Busca, BuscaSemantica
casos = carregar_casos()
resp = [c for c in casos if c['tipo'] == 'resposta']
buscas = {'lexical (TF-IDF)': Busca(), 'semântica (bge-m3)': BuscaSemantica()}
linhas = []
for nome, b in buscas.items():
    r1 = r3 = 0
    for c in resp:
        ids = [x['id'] for x, _ in b.buscar(c['pergunta'], 3)]
        r1 += ids[0] in c['fontes_ok']
        r3 += any(i in c['fontes_ok'] for i in ids)
    linhas.append({'busca': nome, 'recall@1': f'{r1}/{len(resp)} ({100*r1//len(resp)}%)', 'recall@3': f'{r3}/{len(resp)} ({100*r3//len(resp)}%)'})
pd.DataFrame(linhas)

,busca,recall@1,recall@3
0,lexical (TF-IDF),10/18 (55%),16/18 (88%)
1,semântica (bge-m3),17/18 (94%),17/18 (94%)


## Limiar de recusa
Se a melhor nota da busca for menor que o limiar, o assistente nem chama o modelo e responde que não consta. Teste vários valores: quantas perguntas com resposta continuam passando e quantas sem resposta são barradas.

In [4]:
b = buscas['semântica (bge-m3)']
com = [b.buscar(c['pergunta'], 1)[0][1] for c in resp]
sem = [b.buscar(c['pergunta'], 1)[0][1] for c in casos if c['tipo'] == 'nao_consta']
print('notas das perguntas COM resposta:', sorted(round(x, 2) for x in com))
print('notas das perguntas SEM resposta:', sorted(round(x, 2) for x in sem))
pd.DataFrame([{'limiar': l, 'com resposta que passam': f'{sum(x >= l for x in com)}/{len(com)}', 'sem resposta barradas': f'{sum(x < l for x in sem)}/{len(sem)}'} for l in (0.50, 0.55, 0.58, 0.60, 0.65)])

notas das perguntas COM resposta: [0.57, 0.6, 0.62, 0.64, 0.64, 0.65, 0.65, 0.67, 0.68, 0.68, 0.7, 0.7, 0.71, 0.71, 0.74, 0.75, 0.76, 0.78]
notas das perguntas SEM resposta: [0.42, 0.51, 0.55, 0.56, 0.57, 0.57]


,limiar,com resposta que passam,sem resposta barradas
0,0.50,18/18,1/6
1,0.55,18/18,2/6
2,0.58,17/18,6/6
3,0.60,17/18,6/6
4,0.65,11/18,6/6


**Escolha: 0,58.** Com esse limiar todas as perguntas sem resposta são barradas e só 1 pergunta com resposta é perdida. Para um assistente de segurança é melhor recusar do que inventar. Limite: o valor foi escolhido com as mesmas 24 perguntas do teste, então precisa ser revalidado com perguntas novas.